# Benchmark Suite for World Models

This notebook runs comprehensive benchmarks on synthetic world models to evaluate different aspects of world model quality and capabilities.

## Overview
- Create synthetic world models with controlled properties
- Run comprehensive benchmark suite
- Compare results across models and configurations
- Generate benchmark reports and visualizations

In [ ]:
# Setup and imports
import torch
import numpy as np
from world_model_lens.benchmarks.synthetic_world_models import run_all_benchmarks
from world_model_lens.benchmarks.synthetic_world_models import SyntheticWorldModel, WorldModelBenchmark
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)

print("Benchmark environment setup complete")

## Create Synthetic World Models

Generate synthetic world models with varying properties to test benchmark capabilities.

In [ ]:
# Define synthetic world model configurations
from world_model_lens.benchmarks.synthetic_world_models import ModelConfig

model_configs = {
    "perfect_world_model": ModelConfig(
        name="Perfect World Model",
        observation_dim=64,
        action_dim=8,
        hidden_dim=256,
        stochasticity=0.0,           # Deterministic transitions
        horizon_length=100,
        has_reward_model=True,
        has_terminal_model=True,
        representation_type="ideal"
    ),
    "stochastic_world_model": ModelConfig(
        name="Stochastic World Model",
        observation_dim=64,
        action_dim=8,
        hidden_dim=256,
        stochasticity=0.3,           # Moderate stochasticity
        horizon_length=100,
        has_reward_model=True,
        has_terminal_model=True,
        representation_type="compressed"
    ),
    "short_horizon_model": ModelConfig(
        name="Short-Horizon Model",
        observation_dim=64,
        action_dim=8,
        hidden_dim=128,
        stochasticity=0.2,
        horizon_length=20,           # Limited horizon
        has_reward_model=True,
        has_terminal_model=True,
        representation_type="compressed"
    ),
    "noisy_world_model": ModelConfig(
        name="Noisy World Model",
        observation_dim=64,
        action_dim=8,
        hidden_dim=256,
        stochasticity=0.6,           # High stochasticity
        horizon_length=100,
        has_reward_model=True,
        has_terminal_model=True,
        representation_type="noisy"
    ),
    "abstract_world_model": ModelConfig(
        name="Abstract Representation Model",
        observation_dim=64,
        action_dim=8,
        hidden_dim=256,
        stochasticity=0.1,
        horizon_length=100,
        has_reward_model=True,
        has_terminal_model=True,
        representation_type="abstract"
    )
}

# Create synthetic models
synthetic_models = {}
for key, config in model_configs.items():
    synthetic_models[key] = SyntheticWorldModel(config)
    print(f"Created: {config.name}")

print(f"\nTotal models created: {len(synthetic_models)}")

## Configure Benchmark Suite

Define the benchmarks to run on each synthetic model.

In [ ]:
# Define benchmark configuration
from world_model_lens.benchmarks.synthetic_world_models import BenchmarkConfig

benchmark_config = BenchmarkConfig(
    benchmarks=[
        "transition_accuracy",
        "reward_prediction",
        "imagination_rollout",
        "representation_quality",
        "planner_compatibility",
        "counterfactual_reasoning"
    ],
    num_test_trajectories=50,
    trajectory_length=50,
    num_rollout_steps=30,
    metrics=[ "mse", "mae", "ssim", "kl_divergence" ],
    compute_confidence_intervals=True,
    confidence_level=0.95
)

print("Benchmark configuration:")
print(f"  Number of benchmarks: {len(benchmark_config.benchmarks)}")
print(f"  Test trajectories: {benchmark_config.num_test_trajectories}")
print(f"  Trajectory length: {benchmark_config.trajectory_length}")

## Run All Benchmarks

Execute the complete benchmark suite on all synthetic models.

In [ ]:
# Run benchmarks
print("Running comprehensive benchmarks...")
print("="*60)

all_results = run_all_benchmarks(
    models=synthetic_models,
    config=benchmark_config,
    verbose=True
)

print("\n" + "="*60)
print("Benchmarks completed!")

In [ ]:
# Display summary of results
print("\nBenchmark Results Summary:")
print("="*60)

model_names = list(all_results.keys())
benchmark_names = list(all_results[model_names[0]].keys())

# Create summary table
summary_data = []
for model_name in model_names:
    row = {"Model": model_name}
    for benchmark in benchmark_names:
        if benchmark in all_results[model_name]:
            score = all_results[model_name][benchmark].get("overall_score", 0)
            row[benchmark] = f"{score:.3f}"
    summary_data.append(row)

# Pretty print
for row in summary_data:
    print(f"\n{row['Model']}:")
    for key, value in row.items():
        if key != "Model":
            print(f"  {key}: {value}")

## Visualize Benchmark Results

Create comprehensive visualizations of benchmark results.

In [ ]:
# Create comprehensive benchmark comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Flatten for easier indexing
axes = axes.flatten()

for idx, benchmark in enumerate(benchmark_names):
    ax = axes[idx]
    
    # Extract scores for this benchmark
    scores = []
    model_labels = []
    errors = []
    
    for model_name in model_names:
        if benchmark in all_results[model_name]:
            result = all_results[model_name][benchmark]
            scores.append(result.get("overall_score", 0))
            model_labels.append(model_name.replace("_", "\n"))
            if "ci" in result:
                errors.append(result["ci"] / 2)  # Half CI for error bar
            else:
                errors.append(0)
    
    # Create bar chart with error bars
    colors = plt.cm.viridis(np.linspace(0, 1, len(scores)))
    bars = ax.bar(range(len(scores)), scores, color=colors, yerr=errors, capsize=5)
    ax.set_xticks(range(len(scores)))
    ax.set_xticklabels(model_labels, fontsize=8)
    ax.set_ylabel("Score")
    ax.set_title(benchmark.replace("_", " ").title())
    ax.set_ylim([0, 1.1])
    ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("benchmark_results.png", dpi=150)
plt.show()

In [ ]:
# Create heatmap of all benchmark results
fig, ax = plt.subplots(figsize=(12, 8))

# Build matrix
matrix_data = []
row_labels = []

for model_name in model_names:
    row = []
    for benchmark in benchmark_names:
        if benchmark in all_results[model_name]:
            row.append(all_results[model_name][benchmark].get("overall_score", 0))
        else:
            row.append(0)
    matrix_data.append(row)
    row_labels.append(model_configs[model_name].name)

matrix = np.array(matrix_data)

# Create heatmap
sns.heatmap(
    matrix,
    xticklabels=[b.replace("_", "\n").title() for b in benchmark_names],
    yticklabels=row_labels,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    center=0.5,
    vmin=0,
    vmax=1,
    ax=ax
)
ax.set_xlabel("Benchmark")
ax.set_ylabel("Model")
ax.set_title("World Model Benchmark Comparison")

plt.tight_layout()
plt.savefig("benchmark_heatmap.png", dpi=150)
plt.show()

## Analyze Individual Benchmarks

Deep dive into specific benchmark results.

In [ ]:
# Analyze transition accuracy benchmark
print("Transition Accuracy Analysis:")
print("="*60)

for model_name in model_names:
    if "transition_accuracy" in all_results[model_name]:
        result = all_results[model_name]["transition_accuracy"]
        print(f"\n{model_configs[model_name].name}:")
        print(f"  MSE: {result.get('mse', 'N/A')}")
        print(f"  MAE: {result.get('mae', 'N/A')}")
        print(f"  KL Divergence: {result.get('kl_divergence', 'N/A')}")
        print(f"  Overall Score: {result.get('overall_score', 'N/A'):.4f}")

In [ ]:
# Analyze imagination rollout quality
print("\nImagination Rollout Analysis:")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Collect rollout fidelity over horizon
ax1 = axes[0]
for model_name in model_names:
    if "imagination_rollout" in all_results[model_name]:
        result = all_results[model_name]["imagination_rollout"]
        if "rollout_fidelity_by_step" in result:
            steps = list(result["rollout_fidelity_by_step"].keys())
            fidelity = list(result["rollout_fidelity_by_step"].values())
            ax1.plot(steps, fidelity, label=model_configs[model_name].name, linewidth=2)

ax1.set_xlabel("Rollout Step")
ax1.set_ylabel("Fidelity Score")
ax1.set_title("Imagination Fidelity Over Horizon")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Compare final horizon fidelity
ax2 = axes[1]
final_fidelities = []
model_labels = []
for model_name in model_names:
    if "imagination_rollout" in all_results[model_name]:
        result = all_results[model_name]["imagination_rollout"]
        final_fidelities.append(result.get("final_horizon_fidelity", 0))
        model_labels.append(model_configs[model_name].name)

colors = plt.cm.RdYlGn(np.array(final_fidelities))
ax2.barh(model_labels, final_fidelities, color=colors)
ax2.set_xlabel("Final Horizon Fidelity")
ax2.set_title("Imagination Quality at Final Horizon")
ax2.set_xlim([0, 1])

plt.tight_layout()
plt.savefig("imagination_rollout_analysis.png", dpi=150)
plt.show()

## Generate Benchmark Report

Create a comprehensive report summarizing all benchmark results.

In [ ]:
import json
from datetime import datetime

# Generate comprehensive report
report = {
    "metadata": {
        "timestamp": datetime.now().isoformat(),
        "num_models": len(model_names),
        "num_benchmarks": len(benchmark_names),
        "benchmark_config": {
            "num_test_trajectories": benchmark_config.num_test_trajectories,
            "trajectory_length": benchmark_config.trajectory_length,
            "num_rollout_steps": benchmark_config.num_rollout_steps
        }
    },
    "model_summaries": {},
    "benchmark_rankings": {},
    "overall_scores": {}
}

# Calculate overall scores
for model_name in model_names:
    total_score = 0
    count = 0
    for benchmark in benchmark_names:
        if benchmark in all_results[model_name]:
            total_score += all_results[model_name][benchmark].get("overall_score", 0)
            count += 1
    report["overall_scores"][model_name] = total_score / count if count > 0 else 0

# Rank models for each benchmark
for benchmark in benchmark_names:
    scores = []
    for model_name in model_names:
        if benchmark in all_results[model_name]:
            scores.append((model_name, all_results[model_name][benchmark].get("overall_score", 0)))
    scores.sort(key=lambda x: -x[1])
    report["benchmark_rankings"][benchmark] = [s[0] for s in scores]

# Save report
with open("benchmark_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Benchmark Report Generated!")
print("="*60)
print("\nOverall Model Rankings:")
for rank, (model, score) in enumerate(sorted(report['overall_scores'].items(), key=lambda x: -x[1]), 1):
    print(f"  {rank}. {model_configs[model].name}: {score:.4f}")

In [ ]:
# Print detailed report
print("\n" + "="*60)
print("DETAILED BENCHMARK REPORT")
print("="*60)

for benchmark in benchmark_names:
    print(f"\n{benchmark.replace('_', ' ').upper()}:")
    print("-"*40)
    for rank, model_name in enumerate(report["benchmark_rankings"][benchmark], 1):
        score = all_results[model_name][benchmark].get("overall_score", 0)
        print(f"  {rank}. {model_configs[model_name].name}: {score:.4f}")

print("\n" + "="*60)
print("Report saved to benchmark_report.json")